In [17]:
import os
import sys
from pathlib import Path
import certifi
from dotenv import load_dotenv

from langchain_community.chat_models import ChatOllama
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain.tools import tool
import requests

In [18]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()

search_dir = Path.cwd()
search_dirs = (search_dir, *search_dir.parents)
dotenv_path = None
for directory in search_dirs:
    project_root = directory if directory.name == "Langchain-Single-Agent" else directory / "Langchain-Single-Agent"
    project_env = project_root / ".env"
    if project_env.is_file():
        dotenv_path = project_env
        break

if dotenv_path is None:
    for directory in search_dirs:
        candidate = directory / ".env"
        if candidate.is_file():
            dotenv_path = candidate
            project_root = directory
            break

if dotenv_path is None:
    raise FileNotFoundError("Could not find .env. Create it in the Langchain-Single-Agent project folder.")

sys.path.insert(0, str(project_root))
from ollama_agent import invoke_ollama_agent

load_dotenv(dotenv_path=dotenv_path, override=True)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [ ]:
if TAVILY_API_KEY:
    search_tool = TavilySearchResults(
        api_wrapper=TavilySearchAPIWrapper(tavily_api_key=TAVILY_API_KEY),
        max_results=2
    )
else:
    @tool
    def search_tool(query: str) -> str:
        """Search the web for information."""
        return "Web search is unavailable because TAVILY_API_KEY is not configured."

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [ ]:
result = search_tool.invoke("Give me the latest news on AI")
result

In [ ]:
# ==========================================
# LOCAL OLLAMA LLM
# ==========================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0
)

In [ ]:
response = llm.invoke("Tell me a joke about AI")
response 

In [ ]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [ ]:
# ==========================================
# RUN
# ==========================================

response = invoke_ollama_agent(
    "Find the capital of India and then find its current weather.",
    tools=tools,
)


In [ ]:
print(response)